# Logistic Correction Benchmark

Fits the exact form:

$$\text{logit}(p_\text{raw}) = \text{logit}(p_\text{elo}) + \beta^\top x$$

Walk-forward OOF 2020-2024, L2-regularized, nested α tuning on Y-1.

## 1. Setup

In [31]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('../..').resolve()
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold' / 'stage1' / 'hM7_Linj14_tau150_hT7'
OUT_DIR  = PROJECT_ROOT / 'data' / 'logit_benchmark'
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COL    = 'home_win'
ELO_PROB_COL = 'p_elo'
OOF_YEARS    = list(range(2020, 2025))
ALPHA_GRID   = [1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0]
N_PLAYERS    = 7

print('Gold dir:', GOLD_DIR)
print('Alpha grid:', ALPHA_GRID)

Gold dir: C:\Users\arius\Desktop\kalshi_wnba_bot\data\gold\stage1\hM7_Linj14_tau150_hT7
Alpha grid: [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]


## 2. Helpers

In [32]:
CLIP_EPS = 1e-7


def clip_probs(p):
    return np.clip(p, CLIP_EPS, 1 - CLIP_EPS)


def logit(p):
    p = clip_probs(np.asarray(p, dtype=float))
    return np.log(p / (1 - p))


def log_loss(y, p):
    p = clip_probs(np.asarray(p))
    y = np.asarray(y, dtype=float)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


def accuracy(y, p):
    return np.mean((np.asarray(p) >= 0.5) == np.asarray(y, dtype=bool))


def load_year(year):
    return pd.read_csv(GOLD_DIR / f'game_xgboost_input_{year}_REGPST.csv')


PLAYER_MODEL_FEATURES = [
    'm_ewma_pre', 'q_pre', 'days_since_first_report_pre', 'days_since_last_dnp_pre',
    'consec_dnps_pre', 'played_last_game_pre', 'minutes_last_game_pre',
    'days_since_last_played_pre', 'injury_present_flag_pre',
]
RECENT_FORM_FEATURES = ['net_rtg_ewma_pre', 'efg_ewma_pre', 'tov_pct_ewma_pre', 'orb_pct_ewma_pre', 'ftr_ewma_pre']
STYLE_FEATURES       = ['off_3pa_rate_pre', 'def_3pa_allowed_pre', 'off_2pa_rate_pre', 'def_2pa_allowed_pre', 'off_tov_pct_pre', 'def_forced_tov_pre']
SCHEDULE_FEATURES    = ['days_rest_pre', 'is_b2b_pre', 'games_last_4_days_pre', 'games_last_7_days_pre', 'travel_miles_pre', 'timezone_shift_hours_pre']

def build_feature_cols(n):
    cols = []
    for side in ('home', 'away'):
        for slot in range(1, n + 1):
            for f in PLAYER_MODEL_FEATURES:
                cols.append(f'{side}_p{slot}_{f}')
    for f in RECENT_FORM_FEATURES + STYLE_FEATURES + SCHEDULE_FEATURES:
        cols.append(f'home_{f}')
        cols.append(f'away_{f}')
    return cols


def cold_start_mask(df):
    col = 'home_p1_m_ewma_pre'
    return df[col] != 0 if col in df.columns else pd.Series(True, index=df.index)


def fit_logreg_offset(X_tr, y_tr, off_tr, alpha):
    """
    Minimise L2-regularised log-loss with fixed offset:
        logit(p) = off + intercept + X @ beta
    Returns (intercept, beta).  intercept is NOT regularised.
    """
    n, d = X_tr.shape

    def loss_grad(params):
        intercept, beta = params[0], params[1:]
        eta = off_tr + intercept + X_tr @ beta
        p   = clip_probs(expit(eta))
        err = p - y_tr
        ll  = -np.mean(y_tr * np.log(p) + (1 - y_tr) * np.log(1 - p))
        reg = 0.5 * alpha * np.dot(beta, beta)
        g_int  = np.mean(err)
        g_beta = X_tr.T @ err / n + alpha * beta
        return ll + reg, np.r_[g_int, g_beta]

    res = minimize(loss_grad, np.zeros(d + 1), method='L-BFGS-B', jac=True,
                   options={'maxiter': 300, 'ftol': 1e-10, 'gtol': 1e-6})
    return res.x[0], res.x[1:]


def predict_logreg(X, offset, intercept, beta):
    return clip_probs(expit(offset + intercept + X @ beta))


print('Helpers defined.')

Helpers defined.


## 3. Load data

In [33]:
all_data = {yr: load_year(yr) for yr in range(2015, 2025)}

sample = all_data[2015]
FEATURE_COLS = build_feature_cols(N_PLAYERS)
print(f'{len(FEATURE_COLS)} numeric features')

160 numeric features


## 4. Walk-forward OOF with nested α tuning

For each fold year Y:
1. Build ES-train set (2015..Y-2, cold-start filtered) and ES-val (Y-1)
2. Fit StandardScaler on ES-train X; standardise both ES-train and ES-val
3. For each α in ALPHA_GRID, fit on ES-train and score on ES-val → pick best α
4. Retrain on full 2015..Y-1 (cold-start filtered on 2015) with best α → predict on Y

In [34]:
oof_records = []
fold_meta   = {}

def prep(dfs, cold_start_yr=None):
    parts = []
    for yr, df in dfs:
        if yr == cold_start_yr:
            df = df[cold_start_mask(df)]
        parts.append(df)
    return pd.concat(parts, ignore_index=True)

def scale_and_impute(scaler, X, fit=False):
    """Scale then fill NaN with 0 (= imputed to column mean)."""
    X = scaler.fit_transform(X) if fit else scaler.transform(X)
    return np.nan_to_num(X, nan=0.0)

for Y in OOF_YEARS:
    es_yr     = Y - 1
    train_yrs = list(range(2015, es_yr))

    es_train_df = prep([(yr, all_data[yr]) for yr in train_yrs], cold_start_yr=2015)
    es_val_df   = all_data[es_yr]
    test_df     = all_data[Y]

    scaler   = StandardScaler()
    X_estr   = scale_and_impute(scaler, es_train_df[FEATURE_COLS].values, fit=True)
    X_esval  = scale_and_impute(scaler, es_val_df[FEATURE_COLS].values)
    y_estr   = es_train_df[LABEL_COL].values.astype(float)
    y_esval  = es_val_df[LABEL_COL].values.astype(float)
    off_estr  = es_train_df['base_margin'].values.astype(float)
    off_esval = es_val_df['base_margin'].values.astype(float)

    # --- tune α on ES-val ---
    best_alpha, best_ll = ALPHA_GRID[-1], np.inf   # safe fallback
    for alpha in ALPHA_GRID:
        intercept, beta = fit_logreg_offset(X_estr, y_estr, off_estr, alpha)
        p_val = predict_logreg(X_esval, off_esval, intercept, beta)
        ll    = log_loss(y_esval, p_val)
        if np.isfinite(ll) and ll < best_ll:
            best_ll, best_alpha = ll, alpha

    # --- retrain on 2015..Y-1 with best α ---
    full_df = prep([(yr, all_data[yr]) for yr in range(2015, Y)], cold_start_yr=2015)
    scaler2  = StandardScaler()
    X_full   = scale_and_impute(scaler2, full_df[FEATURE_COLS].values, fit=True)
    y_full   = full_df[LABEL_COL].values.astype(float)
    off_full = full_df['base_margin'].values.astype(float)

    intercept_f, beta_f = fit_logreg_offset(X_full, y_full, off_full, best_alpha)

    X_test   = scale_and_impute(scaler2, test_df[FEATURE_COLS].values)
    off_test = test_df['base_margin'].values.astype(float)
    p_oof    = predict_logreg(X_test, off_test, intercept_f, beta_f)
    oof_ll   = log_loss(test_df[LABEL_COL].values, p_oof)

    fold_meta[Y] = {'best_alpha': best_alpha, 'es_val_ll': round(best_ll, 5), 'oof_ll': round(oof_ll, 5)}
    print(f'Fold {Y}: best_alpha={best_alpha:.0e} | ES-val LL={best_ll:.5f} | OOF LL={oof_ll:.5f}')

    rec = test_df[['game_id', 'game_date', 'home_team_id', 'away_team_id',
                   'season', LABEL_COL, ELO_PROB_COL]].copy()
    rec['p_logreg']  = p_oof
    rec['fold_year'] = Y
    oof_records.append(rec)

oof_df = pd.concat(oof_records, ignore_index=True)
print(f'\nTotal OOF rows: {len(oof_df)}')

Fold 2020: best_alpha=1e+02 | ES-val LL=0.61001 | OOF LL=0.60872
Fold 2021: best_alpha=1e+02 | ES-val LL=0.60872 | OOF LL=0.60919
Fold 2022: best_alpha=1e+00 | ES-val LL=0.60106 | OOF LL=0.61269
Fold 2023: best_alpha=1e+00 | ES-val LL=0.61269 | OOF LL=0.59955
Fold 2024: best_alpha=1e+00 | ES-val LL=0.59955 | OOF LL=0.59127

Total OOF rows: 1117


## 5. Results summary

In [35]:
XGB_MEAN_LL = 0.59787   # tuning3 rank-2 winner (honest CV, 160 features)

# Overall
y   = oof_df[LABEL_COL].values.astype(float)
ll_elo    = log_loss(y, oof_df[ELO_PROB_COL].values)
ll_logreg = log_loss(y, oof_df['p_logreg'].values)
acc_elo    = accuracy(y, oof_df[ELO_PROB_COL].values)
acc_logreg = accuracy(y, oof_df['p_logreg'].values)

summary = pd.DataFrame([
    {'model': 'Elo',        'log_loss': round(ll_elo,    5), 'accuracy': round(acc_elo,    4)},
    {'model': 'LogReg corr','log_loss': round(ll_logreg, 5), 'accuracy': round(acc_logreg, 4)},
    {'model': 'XGBoost*',   'log_loss': round(XGB_MEAN_LL,5),'accuracy': None},
])
print('=== Overall OOF 2020-2024 ===')
print(summary.to_string(index=False))
print('* XGBoost = tuning3 rank-2 winner OOF log-loss (honest CV, 160 features)')

# Per-fold
fold_rows = []
for Y in OOF_YEARS:
    sub  = oof_df[oof_df['fold_year'] == Y]
    y_f  = sub[LABEL_COL].values.astype(float)
    fold_rows.append({
        'fold': Y,
        'n':    len(y_f),
        'elo_ll':    round(log_loss(y_f, sub[ELO_PROB_COL].values), 5),
        'logreg_ll': round(log_loss(y_f, sub['p_logreg'].values),   5),
        'best_alpha': fold_meta[Y]['best_alpha'],
    })
fold_df = pd.DataFrame(fold_rows)
print('\n=== Per-fold ===')
print(fold_df.to_string(index=False))

# Save
oof_df.to_csv(OUT_DIR / 'oof_logreg_preds_2020_2024.csv', index=False)
fold_df.to_csv(OUT_DIR / 'logreg_eval_by_fold.csv', index=False)
summary.to_csv(OUT_DIR / 'logreg_eval_summary.csv', index=False)
print('\nSaved outputs to', OUT_DIR)

=== Overall OOF 2020-2024 ===
      model  log_loss  accuracy
        Elo   0.60216    0.6777
LogReg corr   0.60343    0.6759
   XGBoost*   0.59787       NaN
* XGBoost = tuning3 rank-2 winner OOF log-loss (honest CV, 160 features)

=== Per-fold ===
 fold   n  elo_ll  logreg_ll  best_alpha
 2020 147 0.59431    0.60872       100.0
 2021 209 0.60669    0.60919       100.0
 2022 239 0.61294    0.61269         1.0
 2023 260 0.59912    0.59955         1.0
 2024 262 0.59615    0.59127         1.0

Saved outputs to C:\Users\arius\Desktop\kalshi_wnba_bot\data\logit_benchmark


## 6. Experiment: 160 features + home_elo_pre + away_elo_pre

Does giving the logistic model the raw Elo values (on top of the logit(p_elo) offset already in base_margin) help?

In [36]:
FEATURE_COLS_EXT = FEATURE_COLS + ['home_elo_pre', 'away_elo_pre']
print(f'Extended feature set: {len(FEATURE_COLS_EXT)} cols ({len(FEATURE_COLS)} base + 2 elo)')

oof_ext_records = []

for Y in OOF_YEARS:
    es_yr     = Y - 1
    train_yrs = list(range(2015, es_yr))

    es_train_df = prep([(yr, all_data[yr]) for yr in train_yrs], cold_start_yr=2015)
    es_val_df   = all_data[es_yr]
    test_df     = all_data[Y]

    scaler   = StandardScaler()
    X_estr   = scale_and_impute(scaler, es_train_df[FEATURE_COLS_EXT].values, fit=True)
    X_esval  = scale_and_impute(scaler, es_val_df[FEATURE_COLS_EXT].values)
    y_estr   = es_train_df[LABEL_COL].values.astype(float)
    y_esval  = es_val_df[LABEL_COL].values.astype(float)
    off_estr  = es_train_df['base_margin'].values.astype(float)
    off_esval = es_val_df['base_margin'].values.astype(float)

    best_alpha, best_ll = ALPHA_GRID[-1], np.inf
    for alpha in ALPHA_GRID:
        intercept, beta = fit_logreg_offset(X_estr, y_estr, off_estr, alpha)
        p_val = predict_logreg(X_esval, off_esval, intercept, beta)
        ll    = log_loss(y_esval, p_val)
        if np.isfinite(ll) and ll < best_ll:
            best_ll, best_alpha = ll, alpha

    full_df  = prep([(yr, all_data[yr]) for yr in range(2015, Y)], cold_start_yr=2015)
    scaler2  = StandardScaler()
    X_full   = scale_and_impute(scaler2, full_df[FEATURE_COLS_EXT].values, fit=True)
    y_full   = full_df[LABEL_COL].values.astype(float)
    off_full = full_df['base_margin'].values.astype(float)

    intercept_f, beta_f = fit_logreg_offset(X_full, y_full, off_full, best_alpha)

    X_test   = scale_and_impute(scaler2, test_df[FEATURE_COLS_EXT].values)
    off_test = test_df['base_margin'].values.astype(float)
    p_test   = predict_logreg(X_test, off_test, intercept_f, beta_f)
    oof_ll   = log_loss(test_df[LABEL_COL].values.astype(float), p_test)

    print(f'Fold {Y}: best_alpha={best_alpha:.0e}  OOF LL={oof_ll:.5f}')

    test_out = test_df[['game_id', 'game_date', LABEL_COL, ELO_PROB_COL]].copy()
    test_out['p_logreg_ext'] = p_test
    test_out['fold_year']    = Y
    oof_ext_records.append(test_out)

oof_ext_df = pd.concat(oof_ext_records, ignore_index=True)

Extended feature set: 162 cols (160 base + 2 elo)
Fold 2020: best_alpha=1e+02  OOF LL=0.60872
Fold 2021: best_alpha=1e+02  OOF LL=0.60919
Fold 2022: best_alpha=1e+00  OOF LL=0.61271
Fold 2023: best_alpha=1e+00  OOF LL=0.59943
Fold 2024: best_alpha=1e+00  OOF LL=0.59128


In [37]:
# Merge base and ext predictions for comparison
compare_df = oof_df[['game_id', LABEL_COL, ELO_PROB_COL, 'p_logreg']].merge(
    oof_ext_df[['game_id', 'p_logreg_ext']], on='game_id'
)

y = compare_df[LABEL_COL].values.astype(float)
results = [
    ('Elo',              log_loss(y, compare_df[ELO_PROB_COL].values),   accuracy(y, compare_df[ELO_PROB_COL].values)),
    ('LogReg 160',       log_loss(y, compare_df['p_logreg'].values),     accuracy(y, compare_df['p_logreg'].values)),
    ('LogReg 160+elo',   log_loss(y, compare_df['p_logreg_ext'].values), accuracy(y, compare_df['p_logreg_ext'].values)),
    ('XGBoost (rank-2)', XGB_MEAN_LL, None),
]

print('=== Overall OOF 2020-2024 ===')
print(f'{"model":<22} {"log_loss":>10} {"accuracy":>10}')
print('-' * 44)
for name, ll, acc in results:
    acc_str = f'{acc:.4f}' if acc is not None else '  N/A  '
    print(f'{name:<22} {ll:>10.5f} {acc_str:>10}')

delta = log_loss(y, compare_df['p_logreg_ext'].values) - log_loss(y, compare_df['p_logreg'].values)
print(f'Delta (160+elo vs 160): {delta:+.5f}')

=== Overall OOF 2020-2024 ===
model                    log_loss   accuracy
--------------------------------------------
Elo                       0.60216     0.6777
LogReg 160                0.60343     0.6759
LogReg 160+elo            0.60341     0.6750
XGBoost (rank-2)          0.59787      N/A  
Delta (160+elo vs 160): -0.00002


## 7. Top coefficients (pooled retrain on 2015-2024)

In [38]:
import statistics
alpha_mode = statistics.median(fold_meta[Y]['best_alpha'] for Y in OOF_YEARS)

all_df  = prep([(yr, all_data[yr]) for yr in range(2015, 2025)], cold_start_yr=2015)
scaler3 = StandardScaler()
X_all   = scale_and_impute(scaler3, all_df[FEATURE_COLS].values, fit=True)
y_all   = all_df[LABEL_COL].values.astype(float)
off_all = all_df['base_margin'].values.astype(float)

intercept_all, beta_all = fit_logreg_offset(X_all, y_all, off_all, alpha_mode)

coef_df = (pd.DataFrame({'feature': FEATURE_COLS, 'coef': beta_all})
             .assign(abs_coef=lambda d: d['coef'].abs())
             .sort_values('abs_coef', ascending=False)
             .head(30))
print(f'Intercept: {intercept_all:.4f}  (near 0 = Elo well-calibrated)')
print(f'\nTop 30 features by |coefficient| (α={alpha_mode}):\n')
print(coef_df[['feature', 'coef']].to_string(index=False))

Intercept: 0.1379  (near 0 = Elo well-calibrated)

Top 30 features by |coefficient| (α=1.0):

                            feature      coef
                      home_p1_q_pre  0.021710
 away_p5_days_since_last_played_pre  0.021225
       away_p5_played_last_game_pre -0.020230
                 home_p2_m_ewma_pre  0.018758
                      away_p2_q_pre -0.018733
       home_p6_played_last_game_pre  0.018446
            away_p2_consec_dnps_pre -0.017635
    away_p6_days_since_last_dnp_pre  0.017414
    away_p5_injury_present_flag_pre -0.016719
       home_p2_played_last_game_pre  0.016666
    away_p5_days_since_last_dnp_pre -0.016167
                 away_p6_m_ewma_pre -0.015641
                 home_p7_m_ewma_pre  0.015450
                 home_p4_m_ewma_pre  0.015165
                 home_p3_m_ewma_pre  0.014935
            home_p2_consec_dnps_pre -0.014694
      home_p2_minutes_last_game_pre  0.014666
       home_p4_played_last_game_pre  0.014666
       away_p3_played_last_game_

## 8. Reliability diagram

In [39]:
def reliability_data(y, p, n_bins=10):
    bins    = np.linspace(0, 1, n_bins + 1)
    bin_idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
    mp, mt  = [], []
    for i in range(n_bins):
        m = bin_idx == i
        if m.sum() == 0:
            continue
        mp.append(p[m].mean()); mt.append(y[m].mean())
    return np.array(mp), np.array(mt)


y = oof_df[LABEL_COL].values.astype(float)
fig, ax = plt.subplots(figsize=(7, 6))
for label, p_col, color in [('Elo', ELO_PROB_COL, 'steelblue'),
                              ('LogReg corr', 'p_logreg', 'darkorange')]:
    mp, mt = reliability_data(y, oof_df[p_col].values)
    ax.plot(mp, mt, 'o-', label=f'{label} (LL={log_loss(y, oof_df[p_col].values):.5f})',
            color=color, linewidth=1.8)

ax.plot([0,1],[0,1], 'k--', lw=1, label='Perfect')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Empirical win rate')
ax.set_title('Reliability diagram — Elo vs LogReg correction (OOF 2020-2024)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_DIR / 'reliability_logreg_2020_2024.png', dpi=150)
print('Saved reliability diagram')
plt.show()

Saved reliability diagram
